# FDA Drug Adverse Events: Download the openFDA bulk files

The [openFDA](https://open.fda.gov/data/downloads/) website publishes its datasets as zipped JSON files that can be downloaded manually or programmatically. This notebook downloads every file of the **Human Drug Adverse Event** dataset ([FAERS](https://open.fda.gov/apis/drug/event/)) and unzips it into the lakehouse.

After a successful run the default lakehouse has two folders:

| Folder | Contents |
|---|---|
| `Files/fda_ds` | The zipped files exactly as downloaded from openFDA, named `{quarter}-{file}`, for example `2025q4-drug-event-0001-of-0029.json.zip` |
| `Files/fda_ds_unzipped` | One JSON file per zip, for example `2025q4-drug-event-0001-of-0029.json`. These raw JSON files are the Bronze layer read by `nb_fda_01_silver_load` |

**Size and run time**
- The dataset has about 1,800 files: one folder per quarter since 2004 plus an `all_other` folder.
- The zipped files total about 115 GB and the unzipped JSON about 580 GB, so check the capacity's OneLake storage before running.
- Both steps run in parallel threads on the Spark driver. The run time depends mostly on the openFDA download speed. Running it from a pipeline (or as a scheduled notebook run) avoids keeping an interactive session open.

**Re-running is safe**: files that are already downloaded or unzipped are skipped, and incomplete files from an interrupted run are cleaned up and redone.

## Parameters

Defaults are tuned for the Starter Pool's Medium driver (8 vCores). Downloads mostly wait on the network, so they use more threads than there are cores. Unzipping uses the CPU, so it uses about one thread per core.

In [ ]:
p_quarters = ""                  # comma-separated quarters to process, e.g. "2004q1,2025q4,all_other"; empty = every file
p_max_parallel_downloads = 16    # network-bound; try 24-32 if no 429/503 retries appear, lower it if they do
p_max_parallel_unzips = 8        # CPU-bound; about the number of driver vCores
p_overwrite_existing = False     # True re-downloads and re-unzips files that already exist (use when openFDA republishes)
p_download_folder = "fda_ds"
p_unzip_folder = "fda_ds_unzipped"

## Settings and helpers

Pipelines pass an empty parameter as `None`, so `param()` falls back to the default for blank values. Files are written through the lakehouse mount `/lakehouse/default`, so plain Python file I/O works against OneLake.

In [ ]:
import os
import shutil
import time
import zipfile
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
from threading import local
from urllib.parse import urlparse

import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry


def param(value, default):
    if value is None or (isinstance(value, str) and value.strip() == ""):
        return default
    return value


def as_bool(value):
    return value if isinstance(value, bool) else str(value).strip().lower() in ("true", "1", "yes")


QUARTERS = {q.strip().lower() for q in (param(p_quarters, "") or "").split(",") if q.strip()}
MAX_PARALLEL_DOWNLOADS = int(param(p_max_parallel_downloads, 16))
MAX_PARALLEL_UNZIPS = int(param(p_max_parallel_unzips, 8))
OVERWRITE_EXISTING = as_bool(param(p_overwrite_existing, False))

LAKEHOUSE_ROOT = Path("/lakehouse/default/Files")
DOWNLOAD_DIRECTORY = LAKEHOUSE_ROOT / param(p_download_folder, "fda_ds")
UNZIP_DIRECTORY = LAKEHOUSE_ROOT / param(p_unzip_folder, "fda_ds_unzipped")
DOWNLOAD_DIRECTORY.mkdir(parents=True, exist_ok=True)
UNZIP_DIRECTORY.mkdir(parents=True, exist_ok=True)

METADATA_URL = "https://api.fda.gov/download.json"
MAX_RETRIES = 5
CONNECT_TIMEOUT_SECONDS = 30
READ_TIMEOUT_SECONDS = 300
DOWNLOAD_CHUNK_SIZE = 8 * 1024 * 1024   # 8 MB
UNZIP_CHUNK_SIZE = 16 * 1024 * 1024     # 16 MB
REQUEST_DELAY_SECONDS = 0.25            # spreads out the first requests when all threads start together


def format_rate(total_bytes, seconds):
    gb = total_bytes / 1024**3
    mb_per_second = total_bytes / 1024**2 / seconds if seconds > 0 else 0
    return f"{gb:,.1f} GB in {seconds / 60:,.1f} min ({mb_per_second:,.0f} MB/s)"


print(f"Quarters:          {', '.join(sorted(QUARTERS)) if QUARTERS else 'all'}")
print(f"Parallel downloads: {MAX_PARALLEL_DOWNLOADS}")
print(f"Parallel unzips:    {MAX_PARALLEL_UNZIPS}")
print(f"Overwrite existing: {OVERWRITE_EXISTING}")
print(f"Download folder:    {DOWNLOAD_DIRECTORY}")
print(f"Unzip folder:       {UNZIP_DIRECTORY}")

## Step 1: Read the openFDA file list

openFDA publishes a metadata file, [download.json](https://api.fda.gov/download.json), that lists every downloadable file for every dataset. For the drug adverse event dataset, `results.drug.event.partitions` holds one entry per zip file with its URL. The quarter is taken from the URL folder (for example `.../drug/event/2025q4/drug-event-0001-of-0029.json.zip`) and prefixed to the file name, because file names repeat across quarters.

The cell stops with an error if the metadata can't be read, so later cells never run on an empty list.

In [ ]:
response = requests.get(METADATA_URL, timeout=(CONNECT_TIMEOUT_SECONDS, READ_TIMEOUT_SECONDS))
response.raise_for_status()
download_metadata_json = response.json()

event_metadata = download_metadata_json["results"]["drug"]["event"]
partitions = event_metadata["partitions"]
print(f"openFDA export date: {event_metadata.get('export_date', 'n/a')}")
print(f"Files listed:        {len(partitions):,}")


def build_download_item(partition):
    """Build the source URL and lakehouse file names for one openFDA file."""
    file_url = partition["file"]
    url_path = urlparse(file_url).path.strip("/").split("/")
    file_name, quarter = url_path[-1], url_path[-2]
    zip_name = f"{quarter}-{file_name}"
    return {
        "url": file_url,
        "quarter": quarter.lower(),
        "zip_name": zip_name,
        "zip_path": DOWNLOAD_DIRECTORY / zip_name,
        "json_path": UNZIP_DIRECTORY / (zip_name[:-4] if zip_name.lower().endswith(".zip") else zip_name),
    }


download_items = [build_download_item(p) for p in partitions]
if QUARTERS:
    download_items = [item for item in download_items if item["quarter"] in QUARTERS]
    missing = QUARTERS - {item["quarter"] for item in download_items}
    if missing:
        raise ValueError(f"Quarters not found in the openFDA file list: {sorted(missing)}")

if not download_items:
    raise ValueError("No files to process")
print(f"Files selected:      {len(download_items):,}")

## Step 2: Download the zip files in parallel

Each file is downloaded by one of `p_max_parallel_downloads` threads:
- **One HTTP session per thread** with automatic retries. Timeouts, 429 (Too Many Requests) and 5xx errors are retried up to 5 times with exponential back-off, and the server's `Retry-After` header is honoured.
- **Streaming in 8 MB chunks**, so a large file is never held in memory.
- **Safe writes**: data goes to a `.partial` file that is renamed only after the full file (checked against `Content-Length`) has arrived, so the unzip step never sees a half-downloaded zip.
- **Restartable**: files that already exist are skipped unless `p_overwrite_existing` is `True`.

The cell fails at the end if any file could not be downloaded. Re-run it to retry only the failed files.

In [ ]:
thread_local = local()


def get_http_session():
    """Create one retry-enabled HTTP session per worker thread."""
    if not hasattr(thread_local, "session"):
        retry_policy = Retry(
            total=MAX_RETRIES,
            connect=MAX_RETRIES,
            read=MAX_RETRIES,
            status=MAX_RETRIES,
            backoff_factor=2,
            status_forcelist=[408, 425, 429, 500, 502, 503, 504],
            allowed_methods=frozenset(["GET"]),
            respect_retry_after_header=True,
            raise_on_status=False,
        )
        # Each thread has its own session, so a small pool per session is enough.
        adapter = HTTPAdapter(max_retries=retry_policy, pool_connections=2, pool_maxsize=2)
        session = requests.Session()
        session.mount("https://", adapter)
        session.mount("http://", adapter)
        session.headers.update({"User-Agent": "fabric-samples-healthcare/1.0 (FDA dataset download)"})
        thread_local.session = session
    return thread_local.session


def download_file(item):
    """Stream one file to a .partial file, then rename it once complete."""
    destination_path = item["zip_path"]
    partial_path = Path(str(destination_path) + ".partial")

    if not OVERWRITE_EXISTING and destination_path.exists() and destination_path.stat().st_size > 0:
        return "skipped", destination_path.stat().st_size

    if partial_path.exists():
        partial_path.unlink()

    time.sleep(REQUEST_DELAY_SECONDS)
    try:
        with get_http_session().get(item["url"], stream=True,
                                    timeout=(CONNECT_TIMEOUT_SECONDS, READ_TIMEOUT_SECONDS),
                                    allow_redirects=True) as response:
            response.raise_for_status()
            expected_size = response.headers.get("Content-Length")
            expected_size = int(expected_size) if expected_size else None

            downloaded_size = 0
            with open(partial_path, "wb") as output_file:
                for chunk in response.iter_content(chunk_size=DOWNLOAD_CHUNK_SIZE):
                    if chunk:
                        output_file.write(chunk)
                        downloaded_size += len(chunk)

            if expected_size is not None and downloaded_size != expected_size:
                raise IOError(f"Incomplete download: expected {expected_size:,} bytes, received {downloaded_size:,}")

        os.replace(partial_path, destination_path)
        return "downloaded", downloaded_size
    except Exception:
        if partial_path.exists():
            partial_path.unlink()
        raise


download_counts = {"downloaded": 0, "skipped": 0}
downloaded_bytes = 0
failed_downloads = []
start = time.time()

with ThreadPoolExecutor(max_workers=MAX_PARALLEL_DOWNLOADS) as executor:
    futures = {executor.submit(download_file, item): item for item in download_items}
    for done, future in enumerate(as_completed(futures), start=1):
        item = futures[future]
        try:
            status, size = future.result()
            download_counts[status] += 1
            if status == "downloaded":
                downloaded_bytes += size
            print(f"[{done:,}/{len(futures):,}] {status.upper()}: {item['zip_name']} ({size / 1024**2:,.1f} MB)")
        except Exception as error:
            failed_downloads.append((item["zip_name"], item["url"], repr(error)))
            print(f"[{done:,}/{len(futures):,}] FAILED: {item['zip_name']}: {error!r}")

print("\nDownload summary")
print(f"Downloaded: {download_counts['downloaded']:,}  ({format_rate(downloaded_bytes, time.time() - start)})")
print(f"Skipped:    {download_counts['skipped']:,}")
print(f"Failed:     {len(failed_downloads):,}")

if failed_downloads:
    for name, url, error in failed_downloads[:20]:
        print(f"- {name}\n  URL: {url}\n  Error: {error}")
    raise RuntimeError(f"{len(failed_downloads)} file(s) failed to download; re-run the notebook to retry them")

## Step 3: Unzip the files in parallel

Each zip is expected to hold exactly one JSON file. The cell fails for a zip with zero or several files, so a change in the openFDA format is noticed instead of silently loading the wrong file.

Extraction streams the JSON in 16 MB chunks, so even files of close to 1 GB are never fully loaded into memory. Python's zlib releases the GIL while it decompresses, so the `p_max_parallel_unzips` threads really do use several cores at once. Like the download, output goes to a `.partial` file that is renamed when complete, and JSON files that already exist are skipped.

In [ ]:
def unzip_file(item):
    """Extract the single JSON file from one zip, writing to a .partial file first."""
    output_path = item["json_path"]
    partial_path = Path(str(output_path) + ".partial")

    if not OVERWRITE_EXISTING and output_path.exists() and output_path.stat().st_size > 0:
        return "skipped", output_path.stat().st_size

    if partial_path.exists():
        partial_path.unlink()

    try:
        with zipfile.ZipFile(item["zip_path"], "r") as zip_ref:
            members = [m for m in zip_ref.infolist() if not m.is_dir()]
            if len(members) != 1:
                raise ValueError(f"Expected 1 file in the zip, found {len(members)}: {[m.filename for m in members][:5]}")
            with zip_ref.open(members[0], "r") as source, open(partial_path, "wb") as destination:
                shutil.copyfileobj(source, destination, length=UNZIP_CHUNK_SIZE)

        os.replace(partial_path, output_path)
        return "unzipped", output_path.stat().st_size
    except Exception:
        if partial_path.exists():
            partial_path.unlink()
        raise


unzip_counts = {"unzipped": 0, "skipped": 0}
unzipped_bytes = 0
failed_unzips = []
start = time.time()

with ThreadPoolExecutor(max_workers=MAX_PARALLEL_UNZIPS) as executor:
    futures = {executor.submit(unzip_file, item): item for item in download_items}
    for done, future in enumerate(as_completed(futures), start=1):
        item = futures[future]
        try:
            status, size = future.result()
            unzip_counts[status] += 1
            if status == "unzipped":
                unzipped_bytes += size
            print(f"[{done:,}/{len(futures):,}] {status.upper()}: {item['json_path'].name} ({size / 1024**2:,.1f} MB)")
        except Exception as error:
            failed_unzips.append((item["zip_name"], repr(error)))
            print(f"[{done:,}/{len(futures):,}] FAILED: {item['zip_name']}: {error!r}")

print("\nUnzip summary")
print(f"Unzipped: {unzip_counts['unzipped']:,}  ({format_rate(unzipped_bytes, time.time() - start)} written)")
print(f"Skipped:  {unzip_counts['skipped']:,}")
print(f"Failed:   {len(failed_unzips):,}")

if failed_unzips:
    for name, error in failed_unzips[:20]:
        print(f"- {name}: {error}")
    raise RuntimeError(f"{len(failed_unzips)} file(s) failed to unzip; re-run the notebook to retry them")

## Step 4: Check the result

Confirms that every selected openFDA file has a zip and a JSON file in the lakehouse and that no `.partial` files were left behind, and reports the total size per folder.

In [ ]:
missing_zips = [item["zip_name"] for item in download_items if not item["zip_path"].exists()]
missing_json = [item["json_path"].name for item in download_items if not item["json_path"].exists()]
leftover_partials = [p.name for folder in (DOWNLOAD_DIRECTORY, UNZIP_DIRECTORY) for p in folder.glob("*.partial")]

zip_bytes = sum(item["zip_path"].stat().st_size for item in download_items if item["zip_path"].exists())
json_bytes = sum(item["json_path"].stat().st_size for item in download_items if item["json_path"].exists())

print(f"Files expected:     {len(download_items):,}")
print(f"Zip files present:  {len(download_items) - len(missing_zips):,}  ({zip_bytes / 1024**3:,.1f} GB)")
print(f"JSON files present: {len(download_items) - len(missing_json):,}  ({json_bytes / 1024**3:,.1f} GB)")
print(f"Leftover .partial:  {len(leftover_partials):,}")

problems = missing_zips + missing_json + leftover_partials
if problems:
    raise RuntimeError(f"Download is incomplete; first problems: {problems[:10]}")
print("\nAll files downloaded and unzipped.")